<a href="https://colab.research.google.com/github/srujith2006/ObjectDetection/blob/main/Q_videoAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics
!pip install pennylane
!pip install hachoir
!apt-get install ffmpeg -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.4/650.4 kB 11.8 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [2]:
from google.colab import files

uploaded = files.upload()

Saving Injured people detection.v1i.yolov8.zip to Injured people detection.v1i.yolov8.zip


In [4]:
import zipfile

zip_ref = zipfile.ZipFile(
    "/content/Injured people detection.v1i.yolov8.zip",
    "r"
)

zip_ref.extractall("/content/")

zip_ref.close()

print("Dataset Extracted")

Dataset Extracted


In [5]:
import cv2
import torch
import torch.nn as nn
import torch.optim as optim

import pennylane as qml

from ultralytics import YOLO

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader

from PIL import Image

import subprocess
import re

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor()

])

In [12]:
import os
import shutil
import yaml

# Define paths
data_yaml_path = "/content/data.yaml"
source_train_images_dir = "/content/train/images"
source_train_labels_dir = "/content/train/labels"

destination_classification_root = "/content/classification_dataset"
destination_train_dir = os.path.join(destination_classification_root, "train")

# Create the root destination directory if it doesn't exist
os.makedirs(destination_train_dir, exist_ok=True)

# Read data.yaml to get class names
class_names = []
if os.path.exists(data_yaml_path):
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    class_names = data.get('names', [])
    if not class_names:
        print("Warning: Class names not found in data.yaml. Image restructuring might be incomplete.")
else:
    print(f"Warning: data.yaml not found at {data_yaml_path}. Cannot determine class names for restructuring.")

# Create class subdirectories in the destination train directory
for class_name in class_names:
    os.makedirs(os.path.join(destination_train_dir, class_name), exist_ok=True)

print(f"Created class directories in {destination_train_dir}: {class_names}")

# Restructure images based on labels
processed_images_count = 0
if os.path.exists(source_train_images_dir) and os.path.exists(source_train_labels_dir):
    for img_filename in os.listdir(source_train_images_dir):
        # Check for common image extensions
        if img_filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp')):
            base_filename = os.path.splitext(img_filename)[0]
            label_filename = base_filename + '.txt'
            label_filepath = os.path.join(source_train_labels_dir, label_filename)

            image_source_path = os.path.join(source_train_images_dir, img_filename)

            if os.path.exists(label_filepath):
                with open(label_filepath, 'r') as f:
                    labels = f.readlines()

                # For ImageFolder, we assign an image to a single class.
                # If multiple objects/classes are present in a YOLO label file,
                # this logic will assign the image to the class of the first object found.
                if labels:
                    first_label_line = labels[0].strip().split()
                    if len(first_label_line) > 0 and first_label_line[0].isdigit():
                        class_id = int(first_label_line[0])

                        if 0 <= class_id < len(class_names):
                            class_name = class_names[class_id]
                            destination_class_dir = os.path.join(destination_train_dir, class_name)
                            shutil.copy(image_source_path, destination_class_dir)
                            processed_images_count += 1
                        else:
                            print(f"Warning: Class ID {class_id} out of bounds for image {img_filename} in {label_filepath}. Skipping.")
                    else:
                        print(f"Warning: Malformed label line for image {img_filename} in {label_filepath}. Skipping.")
                else:
                    print(f"Warning: No object labels found in {label_filename} for image {img_filename}. Skipping.")
            else:
                print(f"Warning: No label file found for image {img_filename} at {label_filepath}. Skipping.")
    print(f"Finished restructuring 'train' dataset. Processed {processed_images_count} images for classification.")
else:
    print(f"Source 'train' image or label directories not found. Expected: {source_train_images_dir} and {source_train_labels_dir}")

Created class directories in /content/classification_dataset/train: ['Injured', 'Uninjured']
Finished restructuring 'train' dataset. Processed 112 images for classification.


In [14]:
import os

def list_directory_contents(path):
    print(f"Contents of {path}:")
    if not os.path.exists(path):
        print(f"  Directory does not exist: {path}")
        return
    if not os.path.isdir(path):
        print(f"  Path is not a directory: {path}")
        return

    items = os.listdir(path)
    if not items:
        print("  (Empty)")
    else:
        for item in sorted(items):
            full_path = os.path.join(path, item)
            if os.path.isdir(full_path):
                print(f"  [DIR] {item}/")
            else:
                print(f"  [FILE] {item}")

# Inspect the root of the restructured dataset
classification_root = "/content/classification_dataset"
list_directory_contents(classification_root)

# Inspect the train directory
train_dir = os.path.join(classification_root, "train")
list_directory_contents(train_dir)

# Inspect the class subdirectories
class_names = ['Injured', 'Uninjured'] # Assuming these from data.yaml
for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    list_directory_contents(class_path)
    # To avoid excessive output, print only first 10 files if many exist
    if os.path.exists(class_path) and os.path.isdir(class_path):
        files_in_class = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
        if len(files_in_class) > 10:
            print(f"  ... (and {len(files_in_class) - 10} more files in {class_name})")

Contents of /content/classification_dataset:
  [DIR] train/
Contents of /content/classification_dataset/train:
  [DIR] Injured/
  [DIR] Uninjured/
Contents of /content/classification_dataset/train/Injured:
  [FILE] 02_0001_png.rf.393bef103f19122f276bb095b3b2ae8d.jpg
  [FILE] 02_0003_png.rf.4cf55187da3df0a8b14c36785896e77c.jpg
  [FILE] 02_0004_png.rf.e7945101da5115bd5e4a61ad33071dec.jpg
  [FILE] 02_0006_png.rf.aa5ce75660817237f041b7bbe59f69c0.jpg
  [FILE] 02_0010_png.rf.54d7b2d8eec7dfaa35d8bfe7478332f1.jpg
  [FILE] 02_0011_png.rf.792681d7d6982c246f26046fa112e80f.jpg
  [FILE] 02_0013_png.rf.6de7a873e97ddc9dbe3ab6f5f62ea777.jpg
  [FILE] 02_0015_png.rf.8e3e15b9aad5822a2e7064115b94a0b7.jpg
  [FILE] 02_0016_png.rf.8ef0fa23af64bb2104cdef78f97af837.jpg
  [FILE] 02_0018_png.rf.d45a3dc8dd7244c4d1166c28aa2c5392.jpg
  [FILE] 02_0023_png.rf.eae2e147e3a4a0074cadb438ae76f161.jpg
  [FILE] 02_0035_png.rf.1a8104b0afdee58c3d44930f6d02b1c5.jpg
  [FILE] 02_0038_png.rf.0ebfef978af7596a4c9d48fdefff482b.jpg
 

In [16]:
dataset = datasets.ImageFolder(

    "/content/classification_dataset/train",

    transform=transform

)

loader = DataLoader(

    dataset,

    batch_size=8,

    shuffle=True

)

print("Dataset Loaded")

Dataset Loaded


In [17]:
n_qubits = 4

dev = qml.device(

    "default.qubit",

    wires=n_qubits

)

In [18]:
@qml.qnode(dev, interface="torch")

def quantum_circuit(inputs, weights):

    qml.templates.AngleEmbedding(

        inputs,

        wires=range(n_qubits)

    )

    qml.templates.StronglyEntanglingLayers(

        weights,

        wires=range(n_qubits)

    )

    return [

        qml.expval(qml.PauliZ(i))

        for i in range(n_qubits)

    ]

In [19]:
weight_shapes = {

    "weights": (3, n_qubits, 3)

}

In [20]:
quantum_layer = qml.qnn.TorchLayer(

    quantum_circuit,

    weight_shapes

)

In [21]:
class HQMLModel(nn.Module):

    def __init__(self):

        super().__init__()

        # MobileNet Feature Extractor
        self.feature_extractor = models.mobilenet_v2(

            pretrained=True

        )

        # Remove Final Layer
        self.feature_extractor.classifier = nn.Identity()

        # Reduce Features
        self.fc1 = nn.Linear(

            1280,

            4

        )

        # Quantum Layer
        self.quantum = quantum_layer

        # Final Classifier
        self.fc2 = nn.Linear(

            4,

            2

        )

    def forward(self, x):

        x = self.feature_extractor(x)

        x = self.fc1(x)

        x = self.quantum(x)

        x = self.fc2(x)

        return x

In [22]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

model = HQMLModel().to(device)

print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 26.1MB/s]

HQMLModel(
  (feature_extractor): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96, kernel

In [23]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(

    model.parameters(),

    lr=0.001

)

In [24]:
for epoch in range(5):

    running_loss = 0

    for images, labels in loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(

            outputs,

            labels

        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(

        f"Epoch {epoch+1}"

        f" Loss: {running_loss}"

    )

print("HQML Training Completed")

Epoch 1 Loss: 10.995167553424835
Epoch 2 Loss: 10.49483722448349
Epoch 3 Loss: 10.754798114299774
Epoch 4 Loss: 10.380398571491241
Epoch 5 Loss: 9.727442026138306
HQML Training Completed


In [25]:
torch.save(

    model.state_dict(),

    "hqml_model.pth"

)

print("HQML Model Saved")

HQML Model Saved


In [26]:
uploaded = files.upload()

Saving seedvideo.mp4 to seedvideo.mp4


In [27]:
def get_video_location(video_path):

    try:

        command = [

            "ffprobe",

            "-v", "quiet",

            "-print_format", "json",

            "-show_format",

            video_path

        ]

        result = subprocess.check_output(
            command
        ).decode("utf-8")

        lat_match = re.search(

            r'location.*?([+-]\d+\.\d+)',

            result

        )

        lon_match = re.search(

            r'([+-]\d+\.\d+)(?!.*[+-]\d+\.\d+)',

            result

        )

        if lat_match and lon_match:

            latitude = lat_match.group(1)

            longitude = lon_match.group(1)

            return latitude, longitude

    except:
        pass

    return "Unknown", "Unknown"

In [28]:
latitude, longitude = get_video_location(

    "/content/input.mp4"

)

print("Latitude:", latitude)

print("Longitude:", longitude)

Latitude: Unknown
Longitude: Unknown


In [31]:
detector = YOLO("yolov8n.pt")


model = HQMLModel().to(device)

model.load_state_dict(

    torch.load("hqml_model.pth")

)

model.eval()


cap = cv2.VideoCapture(

    "/content/seedvideo.mp4" # Corrected input video path

)

frame_width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

frame_height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

fps = int(
    cap.get(cv2.CAP_PROP_FPS)
)


fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(

    "output.mp4",

    fourcc,

    fps,

    (frame_width, frame_height)

)


while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = detector(frame)

    for r in results:

        boxes = r.boxes

        for box in boxes:

            cls = int(box.cls[0])

            # Detect only PERSON
            if cls != 0:
                continue

            x1, y1, x2, y2 = map(

                int,

                box.xyxy[0]

            )

            # Crop Person
            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # Convert to PIL
            img = Image.fromarray(crop)

            # Transform
            img = transform(img)

            img = img.unsqueeze(0)

            img = img.to(device)

            # HQML Prediction
            with torch.no_grad():

                output = model(img)

                pred = torch.argmax(

                    output,

                    1

                )

            label = (

                "Injured"

                if pred == 0

                else "Safe"

            )

            # Width and Height
            width = x2 - x1

            height = y2 - y1

            # Draw Bounding Box
            cv2.rectangle(

                frame,

                (x1,y1),

                (x2,y2),

                (0,255,0),

                2

            )

            # Display Text
            text = (

                f"Survivor | "

                f"{label} | "

                f"W:{width} "

                f"H:{height} | "

                f"Lat:{latitude} | "

                f"Lon:{longitude}"

            )

            cv2.putText(

                frame,

                text,

                (x1,y1-10),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.5,

                (0,0,255),

                2

            )

    out.write(frame)

print("Video Processing Completed")

cap.release()

out.release()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Streaming output truncated to the last 5000 lines.
Speed: 3.8ms preprocess, 102.0ms inference, 0.6ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 train, 93.1ms
Speed: 3.2ms preprocess, 93.1ms inference, 0.8ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 (no detections), 90.7ms
Speed: 2.8ms preprocess, 90.7ms inference, 0.6ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 (no detections), 88.3ms
Speed: 2.7ms preprocess, 88.3ms inference, 0.7ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 (no detections), 91.9ms
Speed: 3.1ms preprocess, 91.9ms inference, 0.8ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 (no detections), 91.4ms
Speed: 3.4ms preprocess, 91.4ms inference, 0.7ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 train, 90.6ms
Speed: 3.6ms preprocess, 90.6ms inference, 1.1ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 train, 89.8ms
Speed: 3.1ms preprocess, 89.8ms inference

In [33]:
from google.colab import files

files.download("output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>